# EfficientNet-B0 Eye ROI Baseline — FAST + STABLE Colab Version

Bu sürüm, önceki notebook'taki doğruluk/izlenebilirlik kurallarını korurken eğitim hızını artırmak için yeniden optimize edilmiştir.

**Başlıca hız değişiklikleri**
- Eye ROI'ler Colab lokal SSD'ye **224×224 RGB olarak önceden resize edilip cache edilir**.
- Eğitim sırasında tekrar `Resize` yapılmaz.
- Lokal SSD kullanıldığı için `DataLoader(num_workers=2, persistent_workers=True)` kullanılır.
- GPU için `channels_last` bellek biçimi kullanılır.
- CUDA AMP (`float16 + GradScaler`) tekrar açılmıştır; önceki non-finite hatasını oluşturan manuel gradient taraması kaldırılmıştır.
- `GradScaler` taşmaları otomatik yönetir; her batch 4 milyon gradienti ayrıca taranmaz.
- CUDA'da `cudnn.benchmark=True`; kesin bit-bit determinism yerine seed tabanlı tekrarlanabilirlik tercih edilir.
- Frozen stage maksimum **3 epoch**, fine-tuning maksimum **8 epoch** ve patience **2** olarak ayarlanmıştır.
- Batch size **64** yapılmıştır. EfficientNet-B0 + 224×224 için Colab T4/L4 sınıfı GPU'larda genellikle uygundur.
- Her batch `tqdm` ile görünür.

**Sabit proje yolları**
- Input: `MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output`
- Metadata: `eye_roi_output/metadata.csv`
- Output: `MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/<run_id>`

> Bu notebook yeni bir run schema kullanır; eski yavaş notebook checkpoint'lerini otomatik resume etmez.

In [1]:

# ============================================================
# CELL 1 — IMPORTS
# ============================================================

import os
import gc
import sys
import json
import math
import time
import random
import shutil
import hashlib
import platform
import subprocess
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import yaml
from PIL import Image, ImageFile

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = False

print("Python      :", platform.python_version())
print("PyTorch     :", torch.__version__)
print("CUDA        :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)


Python      : 3.12.13
PyTorch     : 2.11.0+cpu
CUDA        : False


In [2]:
# ============================================================
# CELL 2 — FAST CONFIGURATION
# ============================================================

RUN_SCHEMA_VERSION = 5

@dataclass(frozen=True)
class Config:
    seed: int = 42
    image_size: int = 224

    # Faster GPU throughput
    batch_size: int = 64
    num_workers: int = 2
    prefetch_factor: int = 2

    # Faster but still sufficient transfer-learning schedule
    frozen_epochs: int = 3
    finetune_epochs: int = 8

    frozen_lr: float = 1e-3
    finetune_lr: float = 2e-5
    weight_decay: float = 1e-4

    early_stopping_patience: int = 2
    scheduler_patience: int = 1

    dropout: float = 0.2

    threshold_min: float = 0.05
    threshold_max: float = 0.95
    threshold_steps: int = 181

    accepted_statuses: Tuple[str, ...] = ("ok", "success")
    negative_label: str = "real"
    positive_label: str = "fake"

    cache_images_locally: bool = True
    verify_cached_images: bool = True

    # AMP is used only on CUDA.
    use_amp: bool = True

    resume_compatible_run: bool = True

CONFIG = Config()

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

AMP_ENABLED = bool(
    CONFIG.use_amp
    and DEVICE.type == "cuda"
)

print(CONFIG)
print("DEVICE     :", DEVICE)
print("AMP_ENABLED:", AMP_ENABLED)

Config(seed=42, image_size=224, batch_size=64, num_workers=2, prefetch_factor=2, frozen_epochs=3, finetune_epochs=8, frozen_lr=0.001, finetune_lr=2e-05, weight_decay=0.0001, early_stopping_patience=2, scheduler_patience=1, dropout=0.2, threshold_min=0.05, threshold_max=0.95, threshold_steps=181, accepted_statuses=('ok', 'success'), negative_label='real', positive_label='fake', cache_images_locally=True, verify_cached_images=True, use_amp=True, resume_compatible_run=True)
DEVICE     : cpu
AMP_ENABLED: False


In [3]:
# ============================================================
# CELL 3 — SEED + GPU PERFORMANCE SETTINGS
# ============================================================

def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG.seed)

# Performance mode:
# Seed is fixed, but exact bit-for-bit determinism is intentionally disabled
# because deterministic kernels can be much slower on Colab GPUs.
if torch.cuda.is_available():
    torch.use_deterministic_algorithms(False)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

    # TF32 helps on Ampere-or-newer GPUs; harmless when unsupported.
    try:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    except Exception:
        pass

print("Seed fixed:", CONFIG.seed)
print("cuDNN benchmark:", (
    torch.backends.cudnn.benchmark
    if torch.backends.cudnn.is_available()
    else None
))

Seed fixed: 42
cuDNN benchmark: None


In [4]:

# ============================================================
# CELL 4 — MOUNT GOOGLE DRIVE
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError as e:
    raise RuntimeError(
        "This notebook is intended for Google Colab."
    ) from e

MY_DRIVE = Path("/content/drive/MyDrive")

if not MY_DRIVE.is_dir():
    raise RuntimeError("Google Drive mount failed.")

print("Google Drive mounted:", MY_DRIVE)


Mounted at /content/drive
Google Drive mounted: /content/drive/MyDrive


In [5]:

# ============================================================
# CELL 5 — FIXED PROJECT PATHS
# ============================================================

DENEY1_ROOT = (
    MY_DRIVE
    / "AISC DeepFake Çalışmaları"
    / "Deneyler"
    / "Kader"
    / "Deney 1"
)

GOZ_ROOT = DENEY1_ROOT / "Göz"
ROI_ROOT = GOZ_ROOT / "eye_roi_output"
METADATA_PATH = ROI_ROOT / "metadata.csv"

# Output is directly under Kader / Deney 1 / Sonuçlar
RESULTS_ROOT = DENEY1_ROOT / "Sonuçlar"

required_directories = {
    "DENEY1_ROOT": DENEY1_ROOT,
    "GOZ_ROOT": GOZ_ROOT,
    "ROI_ROOT": ROI_ROOT,
}

for name, path in required_directories.items():
    if not path.is_dir():
        raise FileNotFoundError(
            f"{name} was not found:\n{path}"
        )

if not METADATA_PATH.is_file():
    raise FileNotFoundError(
        f"metadata.csv was not found:\n{METADATA_PATH}"
    )

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("PATH CONFIGURATION PASSED")
print("=" * 80)
print("DENEY1_ROOT :", DENEY1_ROOT)
print("GOZ_ROOT    :", GOZ_ROOT)
print("ROI_ROOT    :", ROI_ROOT)
print("METADATA    :", METADATA_PATH)
print("RESULTS_ROOT:", RESULTS_ROOT)


PATH CONFIGURATION PASSED
DENEY1_ROOT : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1
GOZ_ROOT    : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz
ROI_ROOT    : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output
METADATA    : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output/metadata.csv
RESULTS_ROOT: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar


In [6]:

# ============================================================
# CELL 6 — RUN DIRECTORY + SAFE RESUME POLICY
# ============================================================

RUN_SUFFIX = "eye_efficientnet_b0_fast_seed42"
RUN_SCHEMA_FILE = "run_schema.json"
RUN_COMPLETE_FILE = "RUN_COMPLETE.json"

def is_compatible_incomplete_run(path: Path) -> bool:
    schema_path = path / RUN_SCHEMA_FILE
    complete_path = path / RUN_COMPLETE_FILE

    if not path.is_dir():
        return False

    if complete_path.exists():
        return False

    if not schema_path.is_file():
        return False

    try:
        with open(schema_path, "r", encoding="utf-8") as f:
            schema = json.load(f)

        return int(schema.get("schema_version", -1)) == RUN_SCHEMA_VERSION
    except Exception:
        return False


compatible_runs = []

if CONFIG.resume_compatible_run:
    for candidate in RESULTS_ROOT.glob(f"*_{RUN_SUFFIX}*"):
        if is_compatible_incomplete_run(candidate):
            compatible_runs.append(candidate)

compatible_runs.sort(
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if compatible_runs:
    RUN_DIR = compatible_runs[0]
    RUN_ID = RUN_DIR.name
    print("Resuming compatible incomplete run:")
    print(RUN_DIR)
else:
    RUN_ID = (
        datetime.now().strftime("%Y%m%d_%H%M")
        + f"_{RUN_SUFFIX}"
    )

    RUN_DIR = RESULTS_ROOT / RUN_ID

    retry = 1

    while RUN_DIR.exists():
        RUN_DIR = RESULTS_ROOT / f"{RUN_ID}_r{retry}"
        retry += 1

    RUN_ID = RUN_DIR.name
    RUN_DIR.mkdir(parents=True, exist_ok=False)

    with open(
        RUN_DIR / RUN_SCHEMA_FILE,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            {
                "schema_version": RUN_SCHEMA_VERSION,
                "run_id": RUN_ID,
                "created_at": datetime.now().isoformat(),
            },
            f,
            indent=2,
        )

DIRS = {
    "checkpoints": RUN_DIR / "checkpoints",
    "metrics": RUN_DIR / "metrics",
    "predictions": RUN_DIR / "predictions",
    "figures": RUN_DIR / "figures",
    "artifacts": RUN_DIR / "artifacts",
    "logs": RUN_DIR / "logs",
}

for directory in DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

FROZEN_DIR = DIRS["checkpoints"] / "frozen"
FINETUNE_DIR = DIRS["checkpoints"] / "finetune"

FROZEN_DIR.mkdir(parents=True, exist_ok=True)
FINETUNE_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_ID :", RUN_ID)
print("RUN_DIR:", RUN_DIR)


RUN_ID : 20260808_0841_eye_efficientnet_b0_fast_seed42
RUN_DIR: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/20260808_0841_eye_efficientnet_b0_fast_seed42


In [7]:

# ============================================================
# CELL 7 — ATOMIC I/O
# ============================================================

def atomic_write_bytes(data: bytes, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)

    tmp = target.with_suffix(
        target.suffix + ".tmp"
    )

    with open(tmp, "wb") as f:
        f.write(data)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp, target)


def atomic_write_text(
    text: str,
    target: Path,
    encoding: str = "utf-8",
) -> None:
    atomic_write_bytes(
        text.encode(encoding),
        target,
    )


def atomic_json_dump(
    obj: Any,
    target: Path,
) -> None:
    atomic_write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
            default=str,
        ),
        target,
    )


def atomic_yaml_dump(
    obj: Any,
    target: Path,
) -> None:
    atomic_write_text(
        yaml.safe_dump(
            obj,
            sort_keys=False,
            allow_unicode=True,
        ),
        target,
    )


def atomic_csv_dump(
    df: pd.DataFrame,
    target: Path,
) -> None:
    atomic_write_bytes(
        df.to_csv(index=False).encode("utf-8"),
        target,
    )


def atomic_torch_save(
    state: dict,
    target: Path,
) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)

    tmp = target.with_suffix(
        target.suffix + ".tmp"
    )

    torch.save(
        state,
        tmp,
    )

    # Read-back validation on CPU.
    verify = torch.load(
        tmp,
        map_location="cpu",
        weights_only=False,
    )

    required_keys = {
        "checkpoint_schema_version",
        "epoch",
        "model_state_dict",
        "optimizer_state_dict",
        "stage_name",
    }

    if not required_keys.issubset(
        verify.keys()
    ):
        try:
            tmp.unlink()
        finally:
            raise RuntimeError(
                f"Checkpoint validation failed: {tmp}"
            )

    os.replace(
        tmp,
        target,
    )


def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


atomic_yaml_dump(
    asdict(CONFIG),
    RUN_DIR / "config_resolved.yaml",
)

atomic_json_dump(
    {
        "run_id": RUN_ID,
        "schema_version": RUN_SCHEMA_VERSION,
        "python": platform.python_version(),
        "torch": torch.__version__,
        "device": str(DEVICE),
        "gpu": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else None
        ),
        "platform": platform.platform(),
    },
    RUN_DIR / "environment.json",
)

try:
    freeze = subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"],
        text=True,
    )

    atomic_write_text(
        freeze,
        RUN_DIR / "requirements_lock.txt",
    )
except Exception as e:
    atomic_write_text(
        f"pip freeze failed: {repr(e)}\n",
        RUN_DIR / "requirements_lock.txt",
    )

print("Atomic I/O helpers ready.")


Atomic I/O helpers ready.


In [8]:

# ============================================================
# CELL 8 — LOAD + VALIDATE EYE ROI METADATA
# ============================================================

REQUIRED_METADATA_COLUMNS = {
    "sample_id",
    "label",
    "split",
    "status",
    "combined_eye_path",
}

metadata_raw = pd.read_csv(
    METADATA_PATH,
    encoding="utf-8-sig",
)

metadata_raw.columns = [
    str(column).strip()
    for column in metadata_raw.columns
]

missing_columns = (
    REQUIRED_METADATA_COLUMNS
    .difference(
        metadata_raw.columns
    )
)

if missing_columns:
    raise ValueError(
        "metadata.csv is missing required columns: "
        f"{sorted(missing_columns)}"
    )

metadata = metadata_raw.copy()

# Preserve missing values as <NA>; do not convert them to literal "nan".
text_columns = [
    "sample_id",
    "label",
    "split",
    "status",
    "combined_eye_path",
]

for column in text_columns:
    metadata[column] = (
        metadata[column]
        .astype("string")
        .str.strip()
    )

metadata["label"] = (
    metadata["label"]
    .str.lower()
)

metadata["split"] = (
    metadata["split"]
    .str.lower()
)

metadata["status"] = (
    metadata["status"]
    .str.lower()
)

allowed_labels = {
    CONFIG.negative_label,
    CONFIG.positive_label,
}

allowed_splits = {
    "train",
    "val",
    "test",
}

accepted_statuses = {
    str(value).lower()
    for value in CONFIG.accepted_statuses
}

observed_labels = set(
    metadata["label"]
    .dropna()
    .unique()
)

observed_splits = set(
    metadata["split"]
    .dropna()
    .unique()
)

bad_labels = sorted(
    observed_labels
    - allowed_labels
)

bad_splits = sorted(
    observed_splits
    - allowed_splits
)

if bad_labels:
    raise ValueError(
        f"Unexpected labels: {bad_labels}"
    )

if bad_splits:
    raise ValueError(
        f"Unexpected splits: {bad_splits}"
    )

status_counts = (
    metadata["status"]
    .fillna("<missing>")
    .value_counts(
        dropna=False
    )
)

accepted_mask = (
    metadata["status"]
    .isin(
        accepted_statuses
    )
)

audit_only = (
    metadata.loc[
        ~accepted_mask
    ]
    .copy()
)

accepted = (
    metadata.loc[
        accepted_mask
    ]
    .copy()
)

if accepted.empty:
    raise RuntimeError(
        "No accepted ROI rows exist."
    )

# Only accepted rows are required to have sample_id.
missing_sample_mask = (
    accepted["sample_id"].isna()
    | accepted["sample_id"].eq("")
)

if missing_sample_mask.any():
    bad = accepted.loc[
        missing_sample_mask,
        [
            "label",
            "split",
            "status",
            "combined_eye_path",
        ],
    ].head(20)

    raise ValueError(
        "Accepted ROI rows with missing sample_id were found:\n"
        f"{bad}"
    )

duplicate_sample_mask = (
    accepted["sample_id"]
    .duplicated(
        keep=False
    )
)

if duplicate_sample_mask.any():
    examples = (
        accepted.loc[
            duplicate_sample_mask,
            "sample_id",
        ]
        .head(20)
        .tolist()
    )

    raise ValueError(
        "Duplicate sample_id values among accepted rows. "
        f"Examples: {examples}"
    )

missing_path_mask = (
    accepted["combined_eye_path"].isna()
    | accepted["combined_eye_path"].eq("")
)

if missing_path_mask.any():
    bad = accepted.loc[
        missing_path_mask,
        [
            "sample_id",
            "label",
            "split",
            "status",
        ],
    ].head(20)

    raise ValueError(
        "Accepted ROI rows with missing combined_eye_path:\n"
        f"{bad}"
    )


def resolve_roi_path(
    rel_or_abs: str,
) -> Path:
    path = Path(
        str(rel_or_abs)
    )

    if path.is_absolute():
        return path

    return (
        ROI_ROOT
        / path
    )


accepted[
    "resolved_eye_path"
] = (
    accepted[
        "combined_eye_path"
    ]
    .map(
        resolve_roi_path
    )
)

accepted[
    "path_exists"
] = (
    accepted[
        "resolved_eye_path"
    ]
    .map(
        lambda path: path.is_file()
    )
)

missing_file_rows = (
    accepted.loc[
        ~accepted["path_exists"]
    ]
    .copy()
)

eligible = (
    accepted.loc[
        accepted["path_exists"]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

if eligible.empty:
    raise RuntimeError(
        "No eligible eye ROI images were found."
    )

# Split + class checks.
split_class_table = pd.crosstab(
    eligible["split"],
    eligible["label"],
)

for split in [
    "train",
    "val",
    "test",
]:
    split_rows = (
        eligible.loc[
            eligible["split"]
            == split
        ]
    )

    if split_rows.empty:
        raise ValueError(
            f"Required split is absent: {split}"
        )

    labels_here = set(
        split_rows["label"]
        .dropna()
        .unique()
    )

    if labels_here != allowed_labels:
        raise ValueError(
            f"Split {split!r} must contain both classes. "
            f"Found: {sorted(labels_here)}"
        )

# Cross-split path leakage.
split_paths = {
    split: set(
        eligible.loc[
            eligible["split"]
            == split,
            "resolved_eye_path",
        ].astype(str)
    )
    for split in [
        "train",
        "val",
        "test",
    ]
}

path_intersections = {
    "train_val": len(
        split_paths["train"]
        & split_paths["val"]
    ),
    "train_test": len(
        split_paths["train"]
        & split_paths["test"]
    ),
    "val_test": len(
        split_paths["val"]
        & split_paths["test"]
    ),
}

if any(
    path_intersections.values()
):
    raise ValueError(
        "Cross-split duplicate ROI paths detected: "
        f"{path_intersections}"
    )

data_accounting = {
    "run_id": RUN_ID,
    "metadata_path": str(
        METADATA_PATH
    ),
    "total_metadata_rows": int(
        len(metadata)
    ),
    "accepted_status_rows": int(
        len(accepted)
    ),
    "audit_only_rows": int(
        len(audit_only)
    ),
    "missing_files_among_accepted": int(
        len(missing_file_rows)
    ),
    "training_eligible_success_count": int(
        len(eligible)
    ),
    "status_counts": {
        str(key): int(value)
        for key, value
        in status_counts.items()
    },
    "split_class_counts": {
        split: {
            label: int(
                (
                    (
                        eligible["split"]
                        == split
                    )
                    & (
                        eligible["label"]
                        == label
                    )
                ).sum()
            )
            for label in sorted(
                allowed_labels
            )
        }
        for split in [
            "train",
            "val",
            "test",
        ]
    },
    "cross_split_path_intersections": (
        path_intersections
    ),
    "true_video_level_leakage_status": (
        "NOT_VERIFIABLE_FROM_CURRENT_METADATA"
    ),
    "true_video_level_note": (
        "The current metadata video_id field is not treated as "
        "authoritative original source-video identity."
    ),
}

atomic_json_dump(
    data_accounting,
    RUN_DIR
    / "data_accounting.json",
)

atomic_csv_dump(
    eligible,
    DIRS["artifacts"]
    / "eligible_metadata_before_cache.csv",
)

atomic_csv_dump(
    audit_only,
    DIRS["artifacts"]
    / "skipped_metadata.csv",
)

print("=" * 80)
print("METADATA VALIDATION PASSED")
print("=" * 80)
print(status_counts)
print()
print(split_class_table)
print()
print(
    json.dumps(
        data_accounting,
        indent=2,
        ensure_ascii=False,
    )
)


METADATA VALIDATION PASSED
status
ok         2986
no_face     111
Name: count, dtype: Int64

label  fake  real
split            
test    156   146
train  1191  1197
val     141   155

{
  "run_id": "20260808_0841_eye_efficientnet_b0_fast_seed42",
  "metadata_path": "/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output/metadata.csv",
  "total_metadata_rows": 3097,
  "accepted_status_rows": 2986,
  "audit_only_rows": 111,
  "missing_files_among_accepted": 0,
  "training_eligible_success_count": 2986,
  "status_counts": {
    "ok": 2986,
    "no_face": 111
  },
  "split_class_counts": {
    "train": {
      "fake": 1191,
      "real": 1197
    },
    "val": {
      "fake": 141,
      "real": 155
    },
    "test": {
      "fake": 156,
      "real": 146
    }
  },
  "cross_split_path_intersections": {
    "train_val": 0,
    "train_test": 0,
    "val_test": 0
  },
  "true_video_level_leakage_status": "NOT_VERIFIABLE_FROM_CURRENT_METADATA",
  "true_vide

In [9]:
# ============================================================
# CELL 9 — PRE-RESIZED LOCAL SSD CACHE (224×224)
# ============================================================

LOCAL_CACHE_ROOT = Path(
    f"/content/eye_roi_cache_{CONFIG.image_size}"
)

LOCAL_CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

def verify_image_file(path: Path) -> None:
    with Image.open(path) as image:
        image.verify()


def cache_one_image(
    source: Path,
    sample_id: str,
) -> Path:
    # Cache as high-quality JPEG at final model input size.
    destination = (
        LOCAL_CACHE_ROOT
        / f"{sample_id}.jpg"
    )

    if destination.is_file():
        try:
            if destination.stat().st_size > 0:
                if CONFIG.verify_cached_images:
                    verify_image_file(destination)
                return destination
        except Exception:
            try:
                destination.unlink()
            except FileNotFoundError:
                pass

    tmp = destination.with_suffix(".jpg.tmp")

    try:
        with Image.open(source) as image:
            image = image.convert("RGB")

            if image.size != (
                CONFIG.image_size,
                CONFIG.image_size,
            ):
                image = image.resize(
                    (
                        CONFIG.image_size,
                        CONFIG.image_size,
                    ),
                    resample=Image.Resampling.BICUBIC,
                )

            # PIL requires a recognized extension or explicit format.
            image.save(
                tmp,
                format="JPEG",
                quality=95,
                optimize=False,
            )
    except Exception as e:
        try:
            if tmp.exists():
                tmp.unlink()
        except Exception:
            pass

        raise RuntimeError(
            f"Failed to cache image: {source}"
        ) from e

    if tmp.stat().st_size <= 0:
        raise RuntimeError(
            f"Cached image is empty: {source}"
        )

    if CONFIG.verify_cached_images:
        verify_image_file(tmp)

    os.replace(
        tmp,
        destination,
    )

    return destination


if CONFIG.cache_images_locally:
    cached_paths = []

    iterator = tqdm(
        eligible.itertuples(index=False),
        total=len(eligible),
        desc="Pre-resizing + caching eye ROIs",
    )

    for row in iterator:
        cached = cache_one_image(
            source=Path(row.resolved_eye_path),
            sample_id=str(row.sample_id),
        )

        cached_paths.append(
            str(cached)
        )

    eligible["training_eye_path"] = cached_paths
else:
    eligible["training_eye_path"] = (
        eligible["resolved_eye_path"]
        .astype(str)
    )

missing_cached = [
    path
    for path
    in eligible["training_eye_path"]
    if not Path(path).is_file()
]

if missing_cached:
    raise RuntimeError(
        "Cache validation failed. "
        f"Missing files: {len(missing_cached)}"
    )

atomic_csv_dump(
    eligible,
    DIRS["artifacts"]
    / "eligible_metadata.csv",
)

print("=" * 80)
print("PRE-RESIZED LOCAL CACHE PASSED")
print("=" * 80)
print("Usable images :", len(eligible))
print("Cache root    :", LOCAL_CACHE_ROOT)
print("Cached size   :", f"{CONFIG.image_size}x{CONFIG.image_size}")

Pre-resizing + caching eye ROIs:   0%|          | 0/2986 [00:00<?, ?it/s]

PRE-RESIZED LOCAL CACHE PASSED
Usable images : 2986
Cache root    : /content/eye_roi_cache_224
Cached size   : 224x224


In [10]:
# ============================================================
# CELL 10 — FAST TRANSFORMS + LOCAL-SSD DATALOADERS
# ============================================================

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406,
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225,
]

# Images are already 224×224 in local cache.
# Keep augmentation intentionally light for speed.
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(
        p=0.5
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD,
    ),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD,
    ),
])

LABEL_TO_INT = {
    CONFIG.negative_label: 0,
    CONFIG.positive_label: 1,
}

INT_TO_LABEL = {
    0: CONFIG.negative_label,
    1: CONFIG.positive_label,
}


class EyeROIDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        transform,
    ):
        self.df = (
            frame
            .reset_index(drop=True)
            .copy()
        )
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(
        self,
        index,
    ):
        row = self.df.iloc[index]
        path = Path(
            row["training_eye_path"]
        )

        try:
            with Image.open(path) as image:
                image = image.convert("RGB")
                tensor = self.transform(image)
        except Exception as e:
            raise RuntimeError(
                "Failed to read/transform eye ROI. "
                f"index={index}, path={path}"
            ) from e

        label = torch.tensor(
            LABEL_TO_INT[row["label"]],
            dtype=torch.float32,
        )

        return {
            "image": tensor,
            "label": label,
            "sample_id": str(
                row["sample_id"]
            ),
            "path": str(path),
        }


split_frames = {
    split: (
        eligible.loc[
            eligible["split"] == split
        ]
        .copy()
        .reset_index(drop=True)
    )
    for split in [
        "train",
        "val",
        "test",
    ]
}

datasets = {
    "train": EyeROIDataset(
        split_frames["train"],
        train_transform,
    ),
    "val": EyeROIDataset(
        split_frames["val"],
        eval_transform,
    ),
    "test": EyeROIDataset(
        split_frames["test"],
        eval_transform,
    ),
}

loader_generator = torch.Generator()
loader_generator.manual_seed(
    CONFIG.seed
)


def make_loader(
    dataset,
    *,
    shuffle: bool,
):
    kwargs = dict(
        dataset=dataset,
        batch_size=CONFIG.batch_size,
        shuffle=shuffle,
        num_workers=CONFIG.num_workers,
        pin_memory=(
            DEVICE.type == "cuda"
        ),
        drop_last=False,
    )

    if shuffle:
        kwargs["generator"] = (
            loader_generator
        )

    if CONFIG.num_workers > 0:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"] = (
            CONFIG.prefetch_factor
        )

    return DataLoader(
        **kwargs
    )


loaders = {
    "train": make_loader(
        datasets["train"],
        shuffle=True,
    ),
    "val": make_loader(
        datasets["val"],
        shuffle=False,
    ),
    "test": make_loader(
        datasets["test"],
        shuffle=False,
    ),
}

print({
    split: len(loader.dataset)
    for split, loader
    in loaders.items()
})

sample_batch = next(
    iter(loaders["train"])
)

expected_shape = (
    3,
    CONFIG.image_size,
    CONFIG.image_size,
)

if tuple(
    sample_batch["image"]
    .shape[1:]
) != expected_shape:
    raise RuntimeError(
        "Unexpected batch image shape: "
        f"{tuple(sample_batch['image'].shape)}"
    )

print(
    "Batch shape:",
    tuple(
        sample_batch["image"].shape
    ),
)

{'train': 2388, 'val': 296, 'test': 302}
Batch shape: (64, 3, 224, 224)


In [11]:

# ============================================================
# CELL 11 — 5-BATCH I/O SPEED SANITY CHECK
# ============================================================

speed_test_batches = 5

start = time.time()
seen = 0

for batch_index, batch in enumerate(
    tqdm(
        loaders["train"],
        total=min(
            speed_test_batches,
            len(loaders["train"]),
        ),
        desc="DataLoader speed test",
    )
):
    _ = batch["image"]

    seen += 1

    if seen >= speed_test_batches:
        break

elapsed = time.time() - start

images_seen = min(
    seen * CONFIG.batch_size,
    len(loaders["train"].dataset),
)

print(
    f"{seen} batches / ~{images_seen} images "
    f"loaded in {elapsed:.2f} seconds."
)

if elapsed > 60:
    print(
        "WARNING: Local data loading is unusually slow. "
        "Training will still run, but inspect Colab disk/runtime health."
    )
else:
    print(
        "DataLoader speed looks healthy."
    )


DataLoader speed test:   0%|          | 0/5 [00:00<?, ?it/s]

5 batches / ~320 images loaded in 1.60 seconds.
DataLoader speed looks healthy.


In [12]:
# ============================================================
# CELL 12 — EFFICIENTNET-B0 MODEL (CHANNELS_LAST)
# ============================================================

def build_model(
    pretrained: bool = True,
) -> nn.Module:
    weights = (
        EfficientNet_B0_Weights.DEFAULT
        if pretrained
        else None
    )

    try:
        model = efficientnet_b0(
            weights=weights
        )
    except Exception as e:
        raise RuntimeError(
            "EfficientNet-B0 pretrained weights could not be loaded."
        ) from e

    in_features = (
        model.classifier[1]
        .in_features
    )

    model.classifier = nn.Sequential(
        nn.Dropout(
            p=CONFIG.dropout
        ),
        nn.Linear(
            in_features,
            1,
        ),
    )

    if DEVICE.type == "cuda":
        model = model.to(
            memory_format=torch.channels_last
        )

    return model


model = build_model(
    pretrained=True
).to(DEVICE)

total_parameters = sum(
    parameter.numel()
    for parameter
    in model.parameters()
)

print(
    "Total parameters:",
    f"{total_parameters:,}",
)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 115MB/s] 


Total parameters: 4,008,829


In [13]:

# ============================================================
# CELL 13 — METRICS
# ============================================================

def safe_roc_auc(
    y_true,
    probabilities,
):
    if len(
        np.unique(
            y_true
        )
    ) < 2:
        return float("nan")

    return float(
        roc_auc_score(
            y_true,
            probabilities,
        )
    )


def safe_average_precision(
    y_true,
    probabilities,
):
    if len(
        np.unique(
            y_true
        )
    ) < 2:
        return float("nan")

    return float(
        average_precision_score(
            y_true,
            probabilities,
        )
    )


def binary_metrics(
    y_true,
    probabilities,
    threshold: float,
) -> Dict[str, float]:
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    predictions = (
        probabilities
        >= threshold
    ).astype(int)

    tn, fp, fn, tp = (
        confusion_matrix(
            y_true,
            predictions,
            labels=[
                0,
                1,
            ],
        )
        .ravel()
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else float("nan")
    )

    return {
        "threshold": float(
            threshold
        ),
        "accuracy": float(
            accuracy_score(
                y_true,
                predictions,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                predictions,
            )
        ),
        "precision": float(
            precision_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "f1": float(
            f1_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "specificity": float(
            specificity
        ),
        "roc_auc": safe_roc_auc(
            y_true,
            probabilities,
        ),
        "average_precision": (
            safe_average_precision(
                y_true,
                probabilities,
            )
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def select_validation_threshold(
    y_true,
    probabilities,
):
    thresholds = np.linspace(
        CONFIG.threshold_min,
        CONFIG.threshold_max,
        CONFIG.threshold_steps,
    )

    rows = [
        binary_metrics(
            y_true,
            probabilities,
            float(threshold),
        )
        for threshold
        in thresholds
    ]

    table = pd.DataFrame(
        rows
    )

    table[
        "distance_to_0_5"
    ] = (
        table["threshold"]
        - 0.5
    ).abs()

    best = (
        table
        .sort_values(
            [
                "f1",
                "balanced_accuracy",
                "distance_to_0_5",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .iloc[0]
    )

    return (
        float(
            best["threshold"]
        ),
        table.drop(
            columns=[
                "distance_to_0_5"
            ]
        ),
    )


In [14]:

# ============================================================
# CELL 14 — SAFE CHECKPOINT + RNG HELPERS
# ============================================================

CHECKPOINT_SCHEMA_VERSION = 3

def get_rng_state() -> dict:
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": (
            torch.get_rng_state()
            .cpu()
        ),
        "loader_generator": (
            loader_generator
            .get_state()
            .cpu()
        ),
    }

    if torch.cuda.is_available():
        state["cuda"] = [
            cuda_state.cpu()
            for cuda_state
            in torch.cuda.get_rng_state_all()
        ]

    return state


def _to_cpu_byte_tensor(
    value,
) -> torch.Tensor:
    if not torch.is_tensor(
        value
    ):
        value = torch.tensor(
            value,
            dtype=torch.uint8,
        )

    return (
        value
        .detach()
        .cpu()
        .to(
            dtype=torch.uint8
        )
    )


def set_rng_state(
    state: Optional[dict],
) -> None:
    if not state:
        return

    python_state = state.get(
        "python"
    )

    if python_state is not None:
        random.setstate(
            python_state
        )

    numpy_state = state.get(
        "numpy"
    )

    if numpy_state is not None:
        np.random.set_state(
            numpy_state
        )

    torch_state = state.get(
        "torch"
    )

    if torch_state is not None:
        torch.set_rng_state(
            _to_cpu_byte_tensor(
                torch_state
            )
        )

    loader_state = state.get(
        "loader_generator"
    )

    if loader_state is not None:
        loader_generator.set_state(
            _to_cpu_byte_tensor(
                loader_state
            )
        )

    cuda_states = state.get(
        "cuda"
    )

    if (
        torch.cuda.is_available()
        and cuda_states is not None
    ):
        safe_states = [
            _to_cpu_byte_tensor(
                cuda_state
            )
            for cuda_state
            in cuda_states
        ]

        if len(
            safe_states
        ) == torch.cuda.device_count():
            torch.cuda.set_rng_state_all(
                safe_states
            )
        else:
            print(
                "WARNING: CUDA RNG device count differs; "
                "CUDA RNG restore skipped."
            )


def save_checkpoint(
    path: Path,
    *,
    epoch: int,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler,
    scaler,
    best_score: float,
    history: List[dict],
    stage_name: str,
) -> None:
    state = {
        "checkpoint_schema_version": (
            CHECKPOINT_SCHEMA_VERSION
        ),
        "run_schema_version": (
            RUN_SCHEMA_VERSION
        ),
        "epoch": int(
            epoch
        ),
        "stage_name": (
            stage_name
        ),
        "model_state_dict": (
            model.state_dict()
        ),
        "optimizer_state_dict": (
            optimizer.state_dict()
        ),
        "scheduler_state_dict": (
            scheduler.state_dict()
            if scheduler is not None
            else None
        ),
        "scaler_state_dict": (
            scaler.state_dict()
            if scaler is not None
            else None
        ),
        "best_metric_score": float(
            best_score
        ),
        "history": history,
        "config": asdict(
            CONFIG
        ),
        "rng_state": get_rng_state(),
    }

    atomic_torch_save(
        state,
        path,
    )


def checkpoint_is_compatible(
    path: Path,
    stage_name: str,
) -> bool:
    if not path.is_file():
        return False

    try:
        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )

        return (
            int(
                checkpoint.get(
                    "checkpoint_schema_version",
                    -1,
                )
            )
            == CHECKPOINT_SCHEMA_VERSION
            and int(
                checkpoint.get(
                    "run_schema_version",
                    -1,
                )
            )
            == RUN_SCHEMA_VERSION
            and checkpoint.get(
                "stage_name"
            )
            == stage_name
        )
    except Exception:
        return False


def load_checkpoint(
    path: Path,
    *,
    stage_name: str,
    model: nn.Module,
    optimizer: Optional[
        torch.optim.Optimizer
    ] = None,
    scheduler=None,
    scaler=None,
) -> dict:
    if not checkpoint_is_compatible(
        path,
        stage_name,
    ):
        raise RuntimeError(
            "Checkpoint is missing, corrupt, or incompatible:\n"
            f"{path}"
        )

    # Always load checkpoint on CPU first.
    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    model.to(
        DEVICE
    )

    if optimizer is not None:
        optimizer.load_state_dict(
            checkpoint[
                "optimizer_state_dict"
            ]
        )

        # Move optimizer tensors to current device.
        for optimizer_state in (
            optimizer.state.values()
        ):
            for key, value in list(
                optimizer_state.items()
            ):
                if torch.is_tensor(
                    value
                ):
                    optimizer_state[
                        key
                    ] = value.to(
                        DEVICE
                    )

    if (
        scheduler is not None
        and checkpoint.get(
            "scheduler_state_dict"
        ) is not None
    ):
        scheduler.load_state_dict(
            checkpoint[
                "scheduler_state_dict"
            ]
        )

    if (
        scaler is not None
        and checkpoint.get(
            "scaler_state_dict"
        ) is not None
    ):
        scaler.load_state_dict(
            checkpoint[
                "scaler_state_dict"
            ]
        )

    set_rng_state(
        checkpoint.get(
            "rng_state"
        )
    )

    return checkpoint


In [15]:
# ============================================================
# CELL 15 — FAST AMP EPOCH RUNNER WITH LIVE PROGRESS
# ============================================================

criterion = nn.BCEWithLogitsLoss()


def make_grad_scaler(
    enabled: bool,
):
    if not enabled:
        return None

    try:
        return torch.amp.GradScaler(
            "cuda",
            enabled=True,
        )
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(
            enabled=True
        )


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    *,
    training: bool,
    optimizer: Optional[
        torch.optim.Optimizer
    ] = None,
    scaler=None,
    frozen_backbone: bool = False,
    description: str = "",
) -> dict:
    if training:
        if optimizer is None:
            raise ValueError(
                "optimizer is required when training=True"
            )

        model.train()

        if frozen_backbone:
            # Feature extractor is genuinely frozen.
            model.features.eval()
            model.classifier.train()
    else:
        model.eval()

    total_loss = 0.0
    processed = 0

    all_labels = []
    all_probabilities = []
    all_sample_ids = []
    all_paths = []

    progress = tqdm(
        loader,
        total=len(loader),
        desc=description,
        leave=False,
    )

    for batch_index, batch in enumerate(
        progress
    ):
        images = batch[
            "image"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        if DEVICE.type == "cuda":
            images = images.contiguous(
                memory_format=torch.channels_last
            )

        labels = batch[
            "label"
        ].to(
            DEVICE,
            non_blocking=True,
        ).float()

        if training:
            optimizer.zero_grad(
                set_to_none=True
            )

        with torch.set_grad_enabled(
            training
        ):
            if AMP_ENABLED:
                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                    enabled=True,
                ):
                    logits = (
                        model(images)
                        .squeeze(1)
                    )

                    loss = criterion(
                        logits.float(),
                        labels,
                    )
            else:
                logits = (
                    model(images)
                    .squeeze(1)
                )

                loss = criterion(
                    logits.float(),
                    labels,
                )

            if not torch.isfinite(
                loss
            ):
                raise FloatingPointError(
                    f"Non-finite loss at batch={batch_index}"
                )

            if training:
                if scaler is not None:
                    scaler.scale(
                        loss
                    ).backward()

                    # Let GradScaler handle overflow by skipping bad optimizer steps.
                    scaler.step(
                        optimizer
                    )
                    scaler.update()
                else:
                    loss.backward()

                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        max_norm=1.0,
                        error_if_nonfinite=True,
                    )

                    optimizer.step()

        probabilities = torch.sigmoid(
            logits.detach().float()
        )

        if not torch.isfinite(
            probabilities
        ).all():
            raise FloatingPointError(
                f"Non-finite probabilities at batch={batch_index}"
            )

        batch_size = images.size(0)

        total_loss += (
            float(
                loss.detach().item()
            )
            * batch_size
        )

        processed += batch_size

        all_labels.extend(
            labels
            .detach()
            .cpu()
            .numpy()
            .astype(int)
            .tolist()
        )

        all_probabilities.extend(
            probabilities
            .cpu()
            .numpy()
            .astype(float)
            .tolist()
        )

        all_sample_ids.extend(
            list(
                batch["sample_id"]
            )
        )

        all_paths.extend(
            list(
                batch["path"]
            )
        )

        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            processed=processed,
        )

    if processed != len(
        loader.dataset
    ):
        raise RuntimeError(
            "Epoch accounting mismatch: "
            f"processed={processed}, "
            f"dataset={len(loader.dataset)}"
        )

    return {
        "loss": (
            total_loss
            / max(processed, 1)
        ),
        "labels": np.asarray(
            all_labels,
            dtype=int,
        ),
        "probabilities": np.asarray(
            all_probabilities,
            dtype=float,
        ),
        "sample_ids": all_sample_ids,
        "paths": all_paths,
        "count": int(processed),
    }

In [16]:
# ============================================================
# CELL 16 — TWO-BATCH FAST SMOKE TEST
# ============================================================

def model_smoke_test(
    model: nn.Module,
) -> None:
    print(
        "Running two-batch AMP-aware smoke test..."
    )

    backup_state = {
        key: value
        .detach()
        .cpu()
        .clone()
        for key, value
        in model.state_dict().items()
    }

    temporary_optimizer = (
        torch.optim.AdamW(
            model.parameters(),
            lr=1e-6,
        )
    )

    temporary_scaler = (
        make_grad_scaler(
            AMP_ENABLED
        )
    )

    model.train()

    seen_batches = 0

    for batch in tqdm(
        loaders["train"],
        total=min(
            2,
            len(loaders["train"]),
        ),
        desc="Smoke test",
        leave=False,
    ):
        images = batch["image"].to(
            DEVICE,
            non_blocking=True,
        )

        if DEVICE.type == "cuda":
            images = images.contiguous(
                memory_format=torch.channels_last
            )

        labels = batch["label"].to(
            DEVICE,
            non_blocking=True,
        ).float()

        temporary_optimizer.zero_grad(
            set_to_none=True
        )

        if AMP_ENABLED:
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=True,
            ):
                logits = model(
                    images
                ).squeeze(1)

                loss = criterion(
                    logits.float(),
                    labels,
                )
        else:
            logits = model(
                images
            ).squeeze(1)

            loss = criterion(
                logits.float(),
                labels,
            )

        if not torch.isfinite(loss):
            raise FloatingPointError(
                "Smoke test produced non-finite loss."
            )

        if temporary_scaler is not None:
            temporary_scaler.scale(
                loss
            ).backward()

            temporary_scaler.step(
                temporary_optimizer
            )

            temporary_scaler.update()
        else:
            loss.backward()
            temporary_optimizer.step()

        seen_batches += 1

        if seen_batches >= 2:
            break

    if seen_batches < 2:
        raise RuntimeError(
            "Smoke test could not obtain two batches."
        )

    model.load_state_dict(
        backup_state
    )

    model.to(DEVICE)

    if DEVICE.type == "cuda":
        model.to(
            memory_format=torch.channels_last
        )

    del (
        backup_state,
        temporary_optimizer,
        temporary_scaler,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(
        "Smoke test PASSED."
    )


model_smoke_test(
    model
)

Running two-batch AMP-aware smoke test...


Smoke test:   0%|          | 0/2 [00:00<?, ?it/s]

Smoke test PASSED.


In [17]:

# ============================================================
# CELL 17 — TRAINING FUNCTION
# ============================================================

def train_stage(
    *,
    model: nn.Module,
    stage_name: str,
    epochs: int,
    lr: float,
    stage_dir: Path,
    frozen_backbone: bool,
    allow_resume: bool,
) -> dict:
    trainable_parameters = [
        parameter
        for parameter
        in model.parameters()
        if parameter.requires_grad
    ]

    if not trainable_parameters:
        raise RuntimeError(
            f"No trainable parameters for stage={stage_name}"
        )

    optimizer = (
        torch.optim.AdamW(
            trainable_parameters,
            lr=lr,
            weight_decay=(
                CONFIG.weight_decay
            ),
        )
    )

    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=(
                CONFIG.scheduler_patience
            ),
            min_lr=1e-7,
        )
    )

    scaler = make_grad_scaler(
        AMP_ENABLED
    )

    last_checkpoint = (
        stage_dir
        / "last.ckpt"
    )

    best_checkpoint = (
        stage_dir
        / "best.ckpt"
    )

    history = []
    start_epoch = 1
    best_score = -float("inf")
    no_improvement_epochs = 0

    if (
        allow_resume
        and checkpoint_is_compatible(
            last_checkpoint,
            stage_name,
        )
    ):
        print(
            f"[{stage_name}] Resuming compatible checkpoint:"
        )

        print(
            last_checkpoint
        )

        checkpoint = load_checkpoint(
            last_checkpoint,
            stage_name=stage_name,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
        )

        history = list(
            checkpoint.get(
                "history",
                [],
            )
        )

        start_epoch = (
            int(
                checkpoint["epoch"]
            )
            + 1
        )

        best_score = float(
            checkpoint.get(
                "best_metric_score",
                -float("inf"),
            )
        )

        if history:
            monitor_scores = [
                row.get(
                    "val_roc_auc",
                    float("nan"),
                )
                for row
                in history
            ]

            safe_scores = [
                score
                if np.isfinite(score)
                else -float("inf")
                for score
                in monitor_scores
            ]

            best_index = int(
                np.argmax(
                    safe_scores
                )
            )

            no_improvement_epochs = max(
                0,
                len(history)
                - best_index
                - 1,
            )

    if start_epoch > epochs:
        print(
            f"[{stage_name}] Stage already complete."
        )

        if not checkpoint_is_compatible(
            best_checkpoint,
            stage_name,
        ):
            raise RuntimeError(
                f"Stage says complete but compatible best.ckpt is missing: "
                f"{best_checkpoint}"
            )

        return {
            "stage": stage_name,
            "history": history,
            "best_score": best_score,
            "best_checkpoint": str(
                best_checkpoint
            ),
            "last_checkpoint": str(
                last_checkpoint
            ),
        }

    for epoch in range(
        start_epoch,
        epochs + 1,
    ):
        epoch_start = time.time()

        train_output = run_epoch(
            model,
            loaders["train"],
            training=True,
            optimizer=optimizer,
            scaler=scaler,
            frozen_backbone=(
                frozen_backbone
            ),
            description=(
                f"{stage_name} train {epoch}/{epochs}"
            ),
        )

        validation_output = run_epoch(
            model,
            loaders["val"],
            training=False,
            scaler=None,
            frozen_backbone=False,
            description=(
                f"{stage_name} val {epoch}/{epochs}"
            ),
        )

        train_metrics = binary_metrics(
            train_output[
                "labels"
            ],
            train_output[
                "probabilities"
            ],
            threshold=0.5,
        )

        validation_metrics = (
            binary_metrics(
                validation_output[
                    "labels"
                ],
                validation_output[
                    "probabilities"
                ],
                threshold=0.5,
            )
        )

        monitor = (
            validation_metrics[
                "roc_auc"
            ]
        )

        if not np.isfinite(
            monitor
        ):
            monitor = (
                validation_metrics[
                    "f1"
                ]
            )

        scheduler.step(
            monitor
        )

        row = {
            "stage": stage_name,
            "epoch": int(
                epoch
            ),
            "lr": float(
                optimizer
                .param_groups[0][
                    "lr"
                ]
            ),
            "train_loss": float(
                train_output[
                    "loss"
                ]
            ),
            "val_loss": float(
                validation_output[
                    "loss"
                ]
            ),
            **{
                f"train_{key}": value
                for key, value
                in train_metrics.items()
            },
            **{
                f"val_{key}": value
                for key, value
                in validation_metrics.items()
            },
            "epoch_seconds": float(
                time.time()
                - epoch_start
            ),
        }

        history.append(
            row
        )

        improved = (
            monitor
            > best_score
            + 1e-8
        )

        if improved:
            best_score = float(
                monitor
            )
            no_improvement_epochs = 0
        else:
            no_improvement_epochs += 1

        # Every completed epoch is resumable.
        save_checkpoint(
            last_checkpoint,
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            best_score=best_score,
            history=history,
            stage_name=stage_name,
        )

        if improved:
            save_checkpoint(
                best_checkpoint,
                epoch=epoch,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                best_score=best_score,
                history=history,
                stage_name=stage_name,
            )

        atomic_csv_dump(
            pd.DataFrame(
                history
            ),
            DIRS["metrics"]
            / f"{stage_name}_training_history.csv",
        )

        print(
            f"[{stage_name}] epoch {epoch:02d}/{epochs} | "
            f"train_loss={row['train_loss']:.5f} | "
            f"val_loss={row['val_loss']:.5f} | "
            f"val_auc={row['val_roc_auc']:.5f} | "
            f"val_f1={row['val_f1']:.5f} | "
            f"lr={row['lr']:.2e} | "
            f"time={row['epoch_seconds']:.1f}s"
        )

        if (
            no_improvement_epochs
            >= CONFIG.early_stopping_patience
        ):
            print(
                f"[{stage_name}] Early stopping after "
                f"{no_improvement_epochs} non-improving epoch(s)."
            )
            break

    if not checkpoint_is_compatible(
        best_checkpoint,
        stage_name,
    ):
        raise RuntimeError(
            f"No compatible best checkpoint created for stage={stage_name}"
        )

    return {
        "stage": stage_name,
        "history": history,
        "best_score": best_score,
        "best_checkpoint": str(
            best_checkpoint
        ),
        "last_checkpoint": str(
            last_checkpoint
        ),
    }


In [ ]:

# ============================================================
# CELL 18 — STAGE 1: FROZEN EFFICIENTNET-B0 BACKBONE
# ============================================================

for parameter in (
    model.features.parameters()
):
    parameter.requires_grad = False

for parameter in (
    model.classifier.parameters()
):
    parameter.requires_grad = True

frozen_trainable = sum(
    parameter.numel()
    for parameter
    in model.parameters()
    if parameter.requires_grad
)

print(
    "Trainable params (frozen):",
    f"{frozen_trainable:,}",
)

frozen_summary = train_stage(
    model=model,
    stage_name="frozen",
    epochs=(
        CONFIG.frozen_epochs
    ),
    lr=(
        CONFIG.frozen_lr
    ),
    stage_dir=(
        FROZEN_DIR
    ),
    frozen_backbone=True,
    allow_resume=True,
)

atomic_json_dump(
    frozen_summary,
    DIRS["metrics"]
    / "frozen_training_summary.json",
)

frozen_summary


Trainable params (frozen): 1,281


frozen train 1/3:   0%|          | 0/38 [00:00<?, ?it/s]

In [ ]:

# ============================================================
# CELL 19 — STAGE 2: FULL FP32 FINE-TUNING
# ============================================================

FROZEN_BEST = (
    FROZEN_DIR
    / "best.ckpt"
)

if not checkpoint_is_compatible(
    FROZEN_BEST,
    "frozen",
):
    raise RuntimeError(
        "Frozen best checkpoint is missing or incompatible."
    )

# Stage 2 initialization always begins from Stage 1 best weights.
frozen_checkpoint = torch.load(
    FROZEN_BEST,
    map_location="cpu",
    weights_only=False,
)

model.load_state_dict(
    frozen_checkpoint[
        "model_state_dict"
    ]
)

model.to(
    DEVICE
)

for parameter in (
    model.parameters()
):
    parameter.requires_grad = True

finetune_trainable = sum(
    parameter.numel()
    for parameter
    in model.parameters()
    if parameter.requires_grad
)

print(
    "Trainable params (fine-tune):",
    f"{finetune_trainable:,}",
)

# Resume Stage 2 only if a compatible Stage 2 checkpoint from THIS notebook/run exists.
finetune_summary = train_stage(
    model=model,
    stage_name="finetune",
    epochs=(
        CONFIG.finetune_epochs
    ),
    lr=(
        CONFIG.finetune_lr
    ),
    stage_dir=(
        FINETUNE_DIR
    ),
    frozen_backbone=False,
    allow_resume=True,
)

atomic_json_dump(
    finetune_summary,
    DIRS["metrics"]
    / "finetune_training_summary.json",
)

finetune_summary


In [ ]:

# ============================================================
# CELL 20 — VALIDATION THRESHOLD SELECTION
# ============================================================

FINAL_BEST_CHECKPOINT = (
    FINETUNE_DIR
    / "best.ckpt"
)

if not checkpoint_is_compatible(
    FINAL_BEST_CHECKPOINT,
    "finetune",
):
    raise RuntimeError(
        "Final best fine-tune checkpoint is missing or incompatible."
    )

final_checkpoint = torch.load(
    FINAL_BEST_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

model.load_state_dict(
    final_checkpoint[
        "model_state_dict"
    ]
)

model.to(
    DEVICE
)

model.eval()

validation_output = run_epoch(
    model,
    loaders["val"],
    training=False,
    description="Final validation",
)

best_threshold, threshold_table = (
    select_validation_threshold(
        validation_output[
            "labels"
        ],
        validation_output[
            "probabilities"
        ],
    )
)

validation_selected_metrics = (
    binary_metrics(
        validation_output[
            "labels"
        ],
        validation_output[
            "probabilities"
        ],
        threshold=(
            best_threshold
        ),
    )
)

atomic_csv_dump(
    threshold_table,
    DIRS["metrics"]
    / "validation_threshold_search.csv",
)

atomic_json_dump(
    validation_selected_metrics,
    DIRS["metrics"]
    / "validation_selected_threshold_metrics.json",
)

print(
    "Validation-selected threshold:",
    best_threshold,
)

print(
    json.dumps(
        validation_selected_metrics,
        indent=2,
    )
)


In [ ]:

# ============================================================
# CELL 21 — FINAL TEST EVALUATION
# ============================================================

test_output = run_epoch(
    model,
    loaders["test"],
    training=False,
    description="Final test",
)

test_metrics = binary_metrics(
    test_output[
        "labels"
    ],
    test_output[
        "probabilities"
    ],
    threshold=(
        best_threshold
    ),
)

test_predictions = pd.DataFrame({
    "sample_id": (
        test_output[
            "sample_ids"
        ]
    ),
    "path": (
        test_output[
            "paths"
        ]
    ),
    "label_int": (
        test_output[
            "labels"
        ]
    ),
    "label": [
        INT_TO_LABEL[
            int(value)
        ]
        for value
        in test_output[
            "labels"
        ]
    ],
    "prob_fake": (
        test_output[
            "probabilities"
        ]
    ),
})

test_predictions[
    "threshold"
] = (
    best_threshold
)

test_predictions[
    "pred_int"
] = (
    test_predictions[
        "prob_fake"
    ].to_numpy()
    >= best_threshold
).astype(int)

test_predictions[
    "prediction"
] = (
    test_predictions[
        "pred_int"
    ]
    .map(
        INT_TO_LABEL
    )
)

test_predictions[
    "correct"
] = (
    test_predictions[
        "label_int"
    ]
    == test_predictions[
        "pred_int"
    ]
)

atomic_csv_dump(
    pd.DataFrame([
        {
            "model": "EfficientNet-B0",
            **test_metrics,
        }
    ]),
    DIRS["metrics"]
    / "final_test_metrics.csv",
)

atomic_csv_dump(
    test_predictions,
    DIRS["predictions"]
    / "test_predictions.csv",
)

print(
    json.dumps(
        test_metrics,
        indent=2,
    )
)


In [ ]:

# ============================================================
# CELL 22 — FIGURES
# ============================================================

def save_figure(
    figure,
    filename: str,
) -> Path:
    path = (
        DIRS["figures"]
        / filename
    )

    figure.savefig(
        path,
        dpi=150,
        bbox_inches="tight",
    )

    plt.close(
        figure
    )

    with Image.open(
        path
    ) as image:
        if min(
            image.size
        ) < 600:
            raise RuntimeError(
                "Figure resolution quality gate failed: "
                f"{path} -> {image.size}"
            )

    return path


frozen_history = pd.read_csv(
    DIRS["metrics"]
    / "frozen_training_history.csv"
)

finetune_history = pd.read_csv(
    DIRS["metrics"]
    / "finetune_training_history.csv"
)

history = pd.concat(
    [
        frozen_history,
        finetune_history,
    ],
    ignore_index=True,
)

history[
    "global_epoch"
] = np.arange(
    1,
    len(history)
    + 1,
)

# Loss
figure, axis = plt.subplots(
    figsize=(10, 6),
    dpi=150,
)

axis.plot(
    history[
        "global_epoch"
    ],
    history[
        "train_loss"
    ],
    label="Training Loss",
)

axis.plot(
    history[
        "global_epoch"
    ],
    history[
        "val_loss"
    ],
    label="Validation Loss",
)

axis.set_title(
    "EfficientNet-B0 Training and Validation Loss"
)

axis.set_xlabel(
    "Global Epoch"
)

axis.set_ylabel(
    "Loss"
)

axis.legend()
axis.grid(
    True,
    alpha=0.25,
)

figure.tight_layout()

save_figure(
    figure,
    "training_validation_loss_curve.png",
)

# Validation metrics
figure, axis = plt.subplots(
    figsize=(10, 6),
    dpi=150,
)

axis.plot(
    history[
        "global_epoch"
    ],
    history[
        "val_roc_auc"
    ],
    label="Validation ROC-AUC",
)

axis.plot(
    history[
        "global_epoch"
    ],
    history[
        "val_f1"
    ],
    label="Validation F1",
)

axis.plot(
    history[
        "global_epoch"
    ],
    history[
        "val_balanced_accuracy"
    ],
    label="Validation Balanced Accuracy",
)

axis.set_title(
    "EfficientNet-B0 Validation Metrics"
)

axis.set_xlabel(
    "Global Epoch"
)

axis.set_ylabel(
    "Score"
)

axis.set_ylim(
    0,
    1,
)

axis.legend()
axis.grid(
    True,
    alpha=0.25,
)

figure.tight_layout()

save_figure(
    figure,
    "validation_metrics_curve.png",
)

y_test = test_output[
    "labels"
]

p_test = test_output[
    "probabilities"
]

# ROC
fpr, tpr, _ = roc_curve(
    y_test,
    p_test,
)

roc_auc = roc_auc_score(
    y_test,
    p_test,
)

figure, axis = plt.subplots(
    figsize=(10, 6),
    dpi=150,
)

axis.plot(
    fpr,
    tpr,
    label=f"EfficientNet-B0 (AUC={roc_auc:.3f})",
)

axis.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Chance",
)

axis.set_title(
    "Test ROC Curve"
)

axis.set_xlabel(
    "False Positive Rate"
)

axis.set_ylabel(
    "True Positive Rate"
)

axis.legend()
axis.grid(
    True,
    alpha=0.25,
)

figure.tight_layout()

save_figure(
    figure,
    "test_roc_curve.png",
)

# PR
precision_curve, recall_curve, _ = (
    precision_recall_curve(
        y_test,
        p_test,
    )
)

average_precision = (
    average_precision_score(
        y_test,
        p_test,
    )
)

figure, axis = plt.subplots(
    figsize=(10, 6),
    dpi=150,
)

axis.plot(
    recall_curve,
    precision_curve,
    label=(
        "EfficientNet-B0 "
        f"(AP={average_precision:.3f})"
    ),
)

axis.set_title(
    "Test Precision-Recall Curve"
)

axis.set_xlabel(
    "Recall"
)

axis.set_ylabel(
    "Precision"
)

axis.legend()
axis.grid(
    True,
    alpha=0.25,
)

figure.tight_layout()

save_figure(
    figure,
    "test_precision_recall_curve.png",
)

# Confusion matrix
matrix = confusion_matrix(
    y_test,
    (
        p_test
        >= best_threshold
    ).astype(int),
    labels=[
        0,
        1,
    ],
)

figure, axis = plt.subplots(
    figsize=(8, 8),
    dpi=150,
)

image = axis.imshow(
    matrix
)

figure.colorbar(
    image,
    ax=axis,
)

axis.set_title(
    f"Test Confusion Matrix (Threshold={best_threshold:.3f})"
)

axis.set_xlabel(
    "Predicted Label"
)

axis.set_ylabel(
    "True Label"
)

axis.set_xticks(
    [
        0,
        1,
    ],
    labels=[
        "Real",
        "Fake",
    ],
)

axis.set_yticks(
    [
        0,
        1,
    ],
    labels=[
        "Real",
        "Fake",
    ],
)

for row_index in range(
    2
):
    for column_index in range(
        2
    ):
        axis.text(
            column_index,
            row_index,
            str(
                matrix[
                    row_index,
                    column_index,
                ]
            ),
            ha="center",
            va="center",
            fontsize=14,
        )

figure.tight_layout()

save_figure(
    figure,
    "test_confusion_matrix.png",
)

# Validation threshold analysis
figure, axis = plt.subplots(
    figsize=(10, 6),
    dpi=150,
)

axis.plot(
    threshold_table[
        "threshold"
    ],
    threshold_table[
        "f1"
    ],
    label="Validation F1",
)

axis.plot(
    threshold_table[
        "threshold"
    ],
    threshold_table[
        "balanced_accuracy"
    ],
    label="Validation Balanced Accuracy",
)

axis.plot(
    threshold_table[
        "threshold"
    ],
    threshold_table[
        "specificity"
    ],
    label="Validation Specificity",
)

axis.axvline(
    best_threshold,
    linestyle="--",
    label=f"Selected={best_threshold:.3f}",
)

axis.set_title(
    "Validation Threshold Analysis"
)

axis.set_xlabel(
    "Threshold"
)

axis.set_ylabel(
    "Score"
)

axis.set_ylim(
    0,
    1,
)

axis.legend()
axis.grid(
    True,
    alpha=0.25,
)

figure.tight_layout()

save_figure(
    figure,
    "validation_threshold_analysis.png",
)

print(
    "Figures saved:",
    DIRS["figures"],
)


In [ ]:

# ============================================================
# CELL 23 — RELOAD / INFERENCE QUALITY GATE
# ============================================================

def inference_reload_test() -> dict:
    reloaded_model = (
        build_model(
            pretrained=False
        )
        .to(DEVICE)
    )

    checkpoint = torch.load(
        FINAL_BEST_CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )

    reloaded_model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    reloaded_model.to(
        DEVICE
    )

    reloaded_model.eval()

    batch = next(
        iter(
            loaders["test"]
        )
    )

    images = (
        batch["image"][
            : min(
                4,
                len(
                    batch["image"]
                ),
            )
        ]
        .to(DEVICE)
    )

    with torch.no_grad():
        probabilities = torch.sigmoid(
            reloaded_model(
                images
            )
            .squeeze(1)
            .float()
        )

    if probabilities.ndim != 1:
        raise RuntimeError(
            "Unexpected inference output shape: "
            f"{tuple(probabilities.shape)}"
        )

    if not torch.isfinite(
        probabilities
    ).all():
        raise FloatingPointError(
            "Reloaded model produced non-finite probabilities."
        )

    if (
        (
            probabilities
            < 0
        )
        | (
            probabilities
            > 1
        )
    ).any():
        raise RuntimeError(
            "Reloaded model produced probabilities outside [0, 1]."
        )

    return {
        "status": "PASSED",
        "checkpoint": str(
            FINAL_BEST_CHECKPOINT
        ),
        "n_samples": int(
            len(
                probabilities
            )
        ),
        "probabilities": (
            probabilities
            .detach()
            .cpu()
            .numpy()
            .astype(float)
            .tolist()
        ),
    }


inference_test = (
    inference_reload_test()
)

atomic_json_dump(
    inference_test,
    DIRS["metrics"]
    / "inference_reload_test.json",
)

print(
    json.dumps(
        inference_test,
        indent=2,
    )
)


In [ ]:

# ============================================================
# CELL 24 — FINAL MANIFEST + RUN SUMMARY + COMPLETE MARKER
# ============================================================

def build_output_manifest(
    root: Path,
) -> pd.DataFrame:
    rows = []

    for path in sorted(
        root.rglob("*")
    ):
        if not path.is_file():
            continue

        # Exclude temporary files if any.
        if path.suffix.endswith(
            ".tmp"
        ):
            continue

        size = path.stat().st_size

        digest = (
            sha256_file(
                path
            )
            if size
            <= 50
            * 1024
            * 1024
            else ""
        )

        rows.append({
            "relative_path": str(
                path.relative_to(
                    root
                )
            ),
            "size_bytes": int(
                size
            ),
            "sha256": digest,
        })

    return pd.DataFrame(
        rows
    )


manifest = build_output_manifest(
    RUN_DIR
)

atomic_csv_dump(
    manifest,
    RUN_DIR
    / "output_manifest.csv",
)

run_summary = {
    "run_id": RUN_ID,
    "run_schema_version": (
        RUN_SCHEMA_VERSION
    ),
    "experiment": (
        "EfficientNet-B0 Eye ROI Transfer Learning Baseline — Fast AMP"
    ),
    "model": (
        "torchvision EfficientNet-B0"
    ),
    "pretrained_weights": (
        "EfficientNet_B0_Weights.DEFAULT"
    ),
    "input": (
        "combined eye ROI"
    ),
    "image_size": (
        CONFIG.image_size
    ),
    "seed": (
        CONFIG.seed
    ),
    "device": str(
        DEVICE
    ),
    "metadata_path": str(
        METADATA_PATH
    ),
    "roi_root": str(
        ROI_ROOT
    ),
    "results_root": str(
        RESULTS_ROOT
    ),
    "run_dir": str(
        RUN_DIR
    ),
    "split_counts": (
        data_accounting[
            "split_class_counts"
        ]
    ),
    "validation_selected_threshold": (
        best_threshold
    ),
    "validation_metrics": (
        validation_selected_metrics
    ),
    "final_test_metrics": (
        test_metrics
    ),
    "inference_reload_test": (
        inference_test[
            "status"
        ]
    ),
    "output_file_count": int(
        len(
            manifest
        )
    ),
}

atomic_json_dump(
    run_summary,
    RUN_DIR
    / "run_summary.json",
)

atomic_json_dump(
    {
        "status": "COMPLETE",
        "run_id": RUN_ID,
        "completed_at": (
            datetime.now()
            .isoformat()
        ),
    },
    RUN_DIR
    / RUN_COMPLETE_FILE,
)

print("=" * 80)
print("EXPERIMENT COMPLETE")
print("=" * 80)

print(
    json.dumps(
        run_summary,
        indent=2,
        ensure_ascii=False,
        default=str,
    )
)

print()
print("FINAL OUTPUT DIRECTORY:")
print(RUN_DIR)
